# Two-Dimensional Cahn-Hilliard Equation

In [ ]:
import jax
import jax.numpy as jnp
from flax import linen as nn
from flax.training import train_state
import optax
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
%matplotlib inline
from functools import partial
import scipy.io
from sklearn.model_selection import train_test_split
import time
import pickle
import random
import torch
import itertools
from scipy.interpolate import griddata
import scipy
import os
from scipy.stats import pearsonr
import matplotlib.ticker as mticker
from Cahn_Hill_FDM import cahn_hill_solver

In [ ]:
from models_fno import FNO2d
from utils_jax import save_model_params, load_model_params
from utils_jax import dataloader

## Data Preparation

In [ ]:
base_path = "/home/rroy13/scr4-sgoswam4/Rajyasri/PyCahnHilliard/cahn_hilliard_dataset.npz"
# dataset = scipy.io.loadmat(os.path.join(base_path, "AllenCahn2D_32.mat"))
dataset = np.load(base_path,allow_pickle = True)
output = dataset['c']
#inputs = jnp.array(inputs)
output = jnp.array(output[0:1000])
del dataset
  # shape: (Ns, Nt, Nx, Ny)

# shape: (Ns, Nt, Nx, Ny)
Ns, Nt, Nx, Ny = output.shape
print(f"Ns: {Ns}, Nt: {Nt}, Nx: {Nx}, Ny: {Ny}")

tt = Nt//3

# Create input-output training pairs
init_timestep = 0
end_timestep = tt

# Build pairs without loop
input_data_NN = output[:, init_timestep:end_timestep, :, :]
output_data_NN = output[:, init_timestep+1:end_timestep+1, :, :]

input_data_NN = input_data_NN.reshape(-1, Nx, Ny)
output_data_NN = output_data_NN.reshape(-1, Nx, Ny)

# Mesh
x = jnp.linspace(0, 1, Nx)
y = jnp.linspace(0, 1, Ny)
X, Y = jnp.meshgrid(x, y, indexing="ij")

ntrain = input_data_NN.shape[0]
X_repeated = jnp.broadcast_to(X, (ntrain, Nx, Ny))
Y_repeated = jnp.broadcast_to(Y, (ntrain, Nx, Ny))

# Channels-last
input_data_NN_mod = jnp.concatenate([
    input_data_NN[..., None],
    X_repeated[..., None],
    Y_repeated[..., None]
], axis=-1)

output_data_NN_mod = output_data_NN[..., None]

print(input_data_NN_mod.shape, output_data_NN_mod.shape)

# Free memory
del input_data_NN, output_data_NN, X_repeated, Y_repeated

In [ ]:
#Separate into train and test datasets
Ntrain = int(0.8*Ns)
perm = jax.random.permutation(jax.random.PRNGKey(0), Ns)

train_idx = perm[:Ntrain]
test_idx = perm[Ntrain:]

train_x = jnp.take(input_data_NN_mod, train_idx, axis=0)
test_x = jnp.take(input_data_NN_mod, test_idx, axis=0)

train_y = jnp.take(output_data_NN_mod, train_idx, axis=0)
test_y = jnp.take(output_data_NN_mod, test_idx, axis=0)

print(f"train_x shape: {train_x.shape}, train_y shape: {train_y.shape}")
print(f"test_x shape: {test_x.shape}, test_y shape: {test_y.shape}")


In [ ]:
folder = os.getcwd()+"/Coupling2D/2D_CH/FNO"
os.makedirs(folder, exist_ok=True)

In [ ]:
modes1 = 32
modes2 = 32

#Create the FNO-2D model object
fno = FNO2d(in_channels = train_x.shape[-1],
            out_channels = train_y.shape[-1],
            modes1 = modes1,
            modes2 = modes2,
            width = 32,
            n_blocks = 4,
            activation = nn.activation.gelu,  
)

model_fn = jax.jit(fno.apply)

In [ ]:
@jax.jit
def create_input_2d(x):
    """
    x: (N, Nx, Ny)
    returns: (N, Nx, Ny, 3)  -> [u, x, y]
    """
    N, Nx, Ny = x.shape

    # Create mesh
    x_lin = jnp.linspace(0, 1, Nx)
    y_lin = jnp.linspace(0, 1, Ny)
    X, Y = jnp.meshgrid(x_lin, y_lin, indexing="ij")  # (Nx, Ny)

    # Repeat for batch
    X_rep = jnp.broadcast_to(X, (N, Nx, Ny))
    Y_rep = jnp.broadcast_to(Y, (N, Nx, Ny))

    # Concatenate along last axis (channels-last)
    x_with_mesh = jnp.concatenate([
        x[..., None],     # (N, Nx, Ny, 1)
        X_rep[..., None], # (N, Nx, Ny, 1)
        Y_rep[..., None]  # (N, Nx, Ny, 1)
    ], axis=-1)

    return x_with_mesh   # (N, Nx, Ny, 3)
# 4th order Runge-Kutta method
@jax.jit
def RK4(params,x):
    dt = 0.01
    # curr_state = x
    # print(x.shape)
    k1 = model_fn(params,x)
    k1 = create_input_2d(k1[...,0])
    # k1 = k1.reshape(k1.shape[0],nx,ny)
    k2 = model_fn(params,x+0.5*dt*k1)
    k2 = create_input_2d(k2[...,0])
    # k2 = k2.reshape(k2.shape[0],nx,ny)
    k3 = model_fn(params,x+0.5*dt*k2)
    k3 = create_input_2d(k3[...,0])
    # k3 = k3.reshape(k3.shape[0],nx,ny)
    k4 = model_fn(params,x+dt*k3)
    k4 = create_input_2d(k4[...,0])
    # k4 = k4_fn(k4.shape[0],nx,ny)
    next_state = x+(dt/6)*(k1+2*k2+2*k3+k4)
    next_state = next_state[...,0]
    next_state = next_state[...,jnp.newaxis]
    print(next_state.shape)
    # next_state = next_state.reshape(next_state.shape[0],nx*ny)
    return next_state

## Residual and Error Estimator (2D Burger)

In [ ]:
@jax.jit
def cahnhill2d_res_error(params, u_curr, eta_hist, W=2.0, M=1.5, kappa=0.5):
    u_curr_mesh = create_input_2d(u_curr)

    u = u_curr_mesh[..., 0]          # u == c, shape (bs, Nx, Ny)
    x = u_curr_mesh[0, :, 0, 1]      # (Nx,)
    y = u_curr_mesh[0, 0, :, 2]      # (Ny,)

    u_curr_t = model_fn(params, u_curr_mesh)
    u_t = u_curr_t[..., 0]

    # Laplacian of u
    u_x = jnp.gradient(u, x, axis=1)
    u_y = jnp.gradient(u, y, axis=2)

    u_xx = jnp.gradient(u_x, x, axis=1)
    u_yy = jnp.gradient(u_y, y, axis=2)

    lap_u = u_xx + u_yy

    # Chemical potential:
    # mu = 2W u(1-u)(1-2u) - kappa * lap_u
    mu = 2.0 * W * u * (1.0 - u) * (1.0 - 2.0 * u) - kappa * lap_u

    # Laplacian of chemical potential
    mu_x = jnp.gradient(mu, x, axis=1)
    mu_y = jnp.gradient(mu, y, axis=2)

    mu_xx = jnp.gradient(mu_x, x, axis=1)
    mu_yy = jnp.gradient(mu_y, y, axis=2)

    lap_mu = mu_xx + mu_yy

    # Cahn-Hilliard residual:
    # u_t - M * lap_mu = 0
    res = u_t - M * lap_mu

    alpha = 0.001
    r = jnp.linalg.norm(res*0.001) / jnp.linalg.norm(u_curr*1000)
    eta = (alpha * r + (1.0 - alpha) * eta_hist)

    return res, eta

## Numerical Experimentation

### Correlation between EMA-based estimator and actual error

In [ ]:
result_dir = "./TI-FNO-params"
filename = f"best_model_params_FNO_TI_2CH_v2.pkl"
best_params = load_model_params(result_dir, filename = filename)

In [ ]:
## All Samples to compute pearson's coefficient
np.random.seed(50)
test_sample = np.random.choice(1000, size=1000, replace=False)

data_test = output
u_pred_model = np.zeros((len(test_sample),Nt, Nx,Ny))
eta_list_final = np.zeros((len(test_sample),Nt-1))
l2error_list_final = np.zeros((len(test_sample),Nt))

for k,ns in enumerate(test_sample[:1000]):
    if k%100 == 0:
        print(ns)
    u_test = data_test[ns:ns+1,:Nt]
    # print(u_test.max())
    initial_u = data_test[ns:ns+1,0,:]
    # print(initial_u.max())
    u_pred_model[k,0,:] = initial_u
    u_curr = initial_u
    eta = 0
    eta_list = []
    for i in range(1, Nt):
        u_curr_in_FNO = create_input_2d(u_curr) #(Ns, in_channels, Nx)
        u_curr_out_FNO = RK4(best_params, u_curr_in_FNO) #(Ns, out_channels, Nx)
        u_curr = u_curr_out_FNO[...,0]
        # print("abc",u_curr.shape)
        res,eta = cahnhill2d_res_error(best_params,u_curr,eta)
        # print(eta)
        eta_list.append(eta)
        u_pred_model[k,i,:] = u_curr
    # print(np.isnan(eta_list).any())    
    eta_list_final[k]=np.array(eta_list)
    l2_error = []
    for i in range(Nt):
        l2_error.append(np.linalg.norm(u_pred_model[k,i,:] - u_test[0,i])/\
                         np.linalg.norm(u_test[0,i,:]))
    l2error_list_final[k] = np.array(l2_error)
    plt.subplot(1,2,1)
    plt.plot(jnp.linspace(0,1,Nt-1),jnp.array(eta_list))
    plt.subplot(1,2,2)
    plt.plot(jnp.linspace(0,1,Nt),jnp.array(l2_error))
    # print(np.isnan(l2error_list_final).any())
    # print(np.isnan(eta_list_final).any()) 
print("Done")    
r_list = [pearsonr(np.array(l2error_list_final[i,1:]),
                   np.array(eta_list_final[i]))[0] for i in range(len(test_sample))]
r_list = np.array(r_list)
r_list = r_list[r_list>=0.8]
print(np.isnan(r_list).any())
plt.figure(figsize = (4,3.5))
plt.hist(r_list,density = True,bins = 80,color = 'deeppink')
# plt.title("Correlation between error estimator and actual error",fontsize = 14)
plt.xlabel(rf"$\rho_{{corr}}$",fontsize = 14)
plt.ylabel("# of samples",fontsize = 14)
plt.grid(which='major', linestyle='-', axis = 'both', linewidth=0.8, alpha=0.8)
plt.grid(which='minor', linestyle='--',axis = 'both', linewidth=0.5, alpha=0.5)
ax = plt.gca()
ax.xaxis.set_major_locator(mticker.MultipleLocator(0.02))
ax.tick_params(axis='x', labelsize=14)
ax.tick_params(axis='y', labelsize=14)

plt.savefig(folder+"/pearson_coeff.pdf",dpi = 300, bbox_inches='tight')
plt.show()

In [ ]:
np.random.seed(50)
test_sample = np.random.choice(1000, size=1000, replace=False)
r_list = [pearsonr(np.array(l2error_list_final[i,1:]),
                   np.array(eta_list_final[i]))[0] for i in range(len(test_sample))]
r_list = np.array(r_list)
r_list = r_list[r_list>=0.944]
print(np.isnan(r_list).any())
plt.figure(figsize = (4,3.5))
plt.hist(r_list,density = True,bins = 80,color = 'deeppink')
# plt.title("Correlation between error estimator and actual error",fontsize = 14)
plt.xlabel(rf"$\rho_{{corr}}$",fontsize = 14)
plt.ylabel("# of samples",fontsize = 14)
plt.grid(which='major', linestyle='-', axis = 'both', linewidth=0.8, alpha=0.8)
plt.grid(which='minor', linestyle='--',axis = 'both', linewidth=0.5, alpha=0.5)
ax = plt.gca()
ax.xaxis.set_major_locator(mticker.MultipleLocator(0.02))
ax.tick_params(axis='x', labelsize=14)
ax.tick_params(axis='y', labelsize=14)

plt.savefig(folder+"/pearson_coeff.pdf",dpi = 300, bbox_inches='tight')
plt.show()

In [ ]:
u_pred_auto = np.load('upred_auto_2d_ch_240.npz')['u_pred_auto']

In [ ]:
u_auto.shape

### Sample-by-sample study between AR-DON, TI-DON and ANCHOR (Ours)

In [ ]:

# test_sample = rng.choice(2500, size=1, replace=False)
test_sample = 240
print(test_sample)
sample_folder = f"/sample_{test_sample}"
os.makedirs(folder+sample_folder, exist_ok=True)
# print(test_sample,output[test_sample].shape)
eta = 0
eta_list = []
st = time.time()
u_pred_model = np.zeros_like(output[test_sample:test_sample+1])# List to store the states over time
print(u_pred_model.shape)
# test_sample = [test_sample]
initial_u = output[test_sample:test_sample+1,0,:]
u_pred_model[:,0] = initial_u
u_curr = initial_u
print(u_pred_model.shape, u_curr.shape)
for i in range(1, Nt):
    u_curr_in_FNO = create_input_2d(u_curr) #(Ns, in_channels, Nx)
    u_curr_out_FNO = RK4(best_params, u_curr_in_FNO) #(Ns, out_channels, Nx)
    u_curr = u_curr_out_FNO[...,0]
    # print(u_curr.shape)
    res,eta = cahnhill2d_res_error(best_params,u_curr,eta)
    # print(eta)
    eta_list.append(eta)
    # Append the predicted state to the list
    u_pred_model[:,i] = u_curr

print("TI-FNO time:",time.time()-st)    
overall_rel_l2_err = jnp.linalg.norm(u_pred_model - output[test_sample])/jnp.linalg.norm(output[test_sample])
print(f"Overall relative L2 error: {overall_rel_l2_err}")

# Plot of L2 error for each time step
l2_error1 = []
# l2_error2 = []
# print(t)
for i in range(Nt):
    l2_error1.append(np.linalg.norm(u_pred_model[:,i,:] - output[test_sample,i,:])/np.linalg.norm(output[test_sample,i,:]))
    # l2_error2.append(np.linalg.norm(u_pred_coup[:,i,:] - u_test[:, i,:])/np.linalg.norm(u_test[:,i,:]))

In [ ]:
t = np.linspace(0,2,Nt)
plt.figure(figsize =(4,3.5))
plt.plot(t[1:],np.array(eta_list),color = 'indigo',lw = 2)
# plt.title(f"EMA-based Error Estimator")#, Sample:{test_sample[0]}")
plt.xlabel("Time",fontsize = 14)
plt.ylabel(r"Error Estimator ($\eta$)",fontsize = 14)
plt.grid(which='major', linestyle='-', axis = 'both', linewidth=0.8, alpha=0.6)
plt.grid(which='minor', linestyle='--',axis = 'both', linewidth=0.5, alpha=0.4)
plt.text(0.1, 0.8, rf"$\rho_{{corr}}$ = {pearsonr(np.array(l2_error1[1:]),np.array(eta_list))[0]:.3f}",
    transform=plt.gca().transAxes,fontsize = 14,va='bottom',
    bbox=dict(
        boxstyle="round,pad=0.3",
        facecolor="white",
        edgecolor="black",
        alpha=0.8))
plt.minorticks_on()
ax = plt.gca()
ax.xaxis.set_major_locator(mticker.MultipleLocator(0.2))
# ax.yaxis.set_major_locator(mticker.MultipleLocator(0.001))
plt.savefig(folder+sample_folder+"/err_estm.pdf",dpi = 300, bbox_inches='tight')

t = np.linspace(0,2,Nt)
plt.figure(figsize =(4,3.5))
plt.plot(t,np.array(l2_error1),color = 'blue',lw = 2)
# plt.title(f"EMA-based Error Estimator")#, Sample:{test_sample[0]}")
plt.xlabel("Time",fontsize = 14)
plt.ylabel(r"Error Estimator ($\eta$)",fontsize = 14)
plt.grid(which='major', linestyle='-', axis = 'both', linewidth=0.8, alpha=0.6)
plt.grid(which='minor', linestyle='--',axis = 'both', linewidth=0.5, alpha=0.4)
plt.minorticks_on()
ax = plt.gca()
ax.xaxis.set_major_locator(mticker.MultipleLocator(0.2))

In [ ]:
eta = 0
# eta_list = []
u_test = output[test_sample:test_sample+1]
u_pred_coup = np.zeros_like(output[test_sample:test_sample+1])# List to store the states over time
initial_u = output[test_sample:test_sample+1,0,:]
# print("initial_u", initial_u.shape)
u_pred_coup[:,0] = initial_u
# print(u_pred_coup.shape)
u_0 = create_input_2d(initial_u)
# Initialize the previous state (this could be your u_0 and u_1, etc.)
u_curr = RK4(best_params,u_0) # Set the current state to the initial state
print("uuu",u_curr.shape) 
i = 1
mark_model = []
mark_coup = []

eta_thres = 0
print(initial_u.shape)
res,eta = cahnhill2d_res_error(best_params,initial_u,eta_hist = 0)

umax0 = np.max(initial_u)
print("umax0:",umax0)
t = np.linspace(0,2,Nt)
st = time.time()
while i<Nt:
    ti = t[i]
    # print("yyy",u_curr.shape)
    res,eta = cahnhill2d_res_error(best_params,u_curr[...,0],eta) # computes residual at ith step using u_curr at ith step
    
    Kut = np.exp(-2*ti)*(np.exp(-umax0)*umax0)+0.1
    # Kut = 10000
    eta_thres = (Kut)
    print(f"estimator:{eta:.6f}|threshold:{eta_thres:.6f}")
    if eta<eta_thres:
        #print(f"tidon {i}")
        # print("u_curr shape:",u_curr.shape)
        u_pred_coup[:,i] = u_curr[0,:,:,0]
        mark_model.append(i)
        i+=1
        u_curr = RK4(best_params,create_input_2d(u_curr[...,0]))  # u_curr at i+1
    else:
        print(f"ns {i}")
        u_init = u_curr[...,0]
        print(u_init.shape)
        dt = 0.01
        nsteps = 41
        tf = (nsteps-1)*dt
        u_final = cahn_hill_solver(u_init[0],tf)
        
        print("u_final",u_final.shape)
        print(u_pred_coup[:,i:i+nsteps,:].shape)
        if i+nsteps<Nt:
            u_pred_coup[:,i:i+nsteps,:] = u_final
        else:
            nsteps = Nt-i
            u_pred_coup[:,i:i+nsteps,:] = u_final[:nsteps,:]
        mark_coup.append(list(range(i, i+nsteps)))
        i+=nsteps
        u_curr = u_pred_coup[:,i-1,:]
        # print("xxx",u_curr.shape)
        res,eta = cahnhill2d_res_error(best_params,u_curr,eta_hist=0)
        u_curr = RK4(best_params,create_input_2d(u_curr))

et = time.time()
print("ANCHOR Time:",et-st)    

print(u_pred_coup.shape)
overall_rel_l2_err = jnp.linalg.norm(u_pred_coup - output[test_sample])/jnp.linalg.norm(output[test_sample])
print(f"Overall relative L2 error: {overall_rel_l2_err}")

In [ ]:
st = time.time()
u_final = cahn_hill_solver(initial_u, tfinal=1.0)
et = time.time()-st
print("Solver Time: ",et)

In [ ]:
x = np.linspace(0, 1, Nx)
y = np.linspace(0, 1, Ny)
X, Y = np.meshgrid(x, y)

tsteps = [0, 40, 80, 120, 160, 200]

nrow = 4
ncol = len(tsteps)

row_labels = [
    "Ground Truth",
    "AR-FNO",
    "TI-FNO",
    "ANCHOR\n(Ours)",
]

# -------------------------
# Color limits per row
# -------------------------
row_clims = {}

# Row 0: Ground truth
row_clims[0] = (
    min(u_test[0][t].min() for t in tsteps),
    max(u_test[0][t].max() for t in tsteps),
)

# Row 1: Autoregressive error
row_clims[1] = (
    min(np.abs(u_pred_auto[0][t] - u_test[0][t]).min() for t in tsteps),
    max(np.abs(u_pred_auto[0][t] - u_test[0][t]).max() for t in tsteps),
)

# Row 2: TIDON error
row_clims[2] = (
    min(np.abs(u_pred_model[0][t] - u_test[0][t]).min() for t in tsteps),
    max(np.abs(u_pred_model[0][t] - u_test[0][t]).max() for t in tsteps),
)

# Row 3: TIDON+NS error
row_clims[3] = (
    min(np.abs(u_pred_coup[0][t] - u_test[0][t]).min() for t in tsteps),
    max(np.abs(u_pred_coup[0][t] - u_test[0][t]).max() for t in tsteps),
)


fig = plt.figure(figsize=(18, 10))

gs = fig.add_gridspec(
    nrow, ncol,
    hspace=0.12,   # ↓ decrease vertical spacing
    wspace=0.16    # ↓ decrease horizontal spacing
)

top_row_axes = []

for i in range(nrow):
    row_axes = []
    im = None

    for j in range(ncol):
        idx = i * ncol + j + 1
        ax = fig.add_subplot(gs[i, j])
        
        vmin, vmax = row_clims[i]

        if i == 0:
            im = ax.pcolormesh(
                X, Y, u_test[0][tsteps[j]],
                cmap='RdBu',
                shading='gouraud',
                vmin=vmin, vmax=vmax
            )

        elif i == 1:
            im = ax.pcolormesh(
                X, Y, np.abs(u_pred_auto[0][tsteps[j]] - u_test[0][tsteps[j]]),
                cmap='bone',
                shading='gouraud',
                vmin=vmin, vmax=vmax
            )

        elif i == 2:
            im = ax.pcolormesh(
                X, Y, np.abs(u_pred_model[0][tsteps[j]] - u_test[0][tsteps[j]]),
                cmap='bone',
                shading='gouraud',
                vmin=vmin, vmax=vmax
            )

        elif i == 3:
            im = ax.pcolormesh(
                X, Y, np.abs(u_pred_coup[0][tsteps[j]] - u_test[0][tsteps[j]]),
                cmap='bone',
                shading='gouraud',
                vmin=vmin, vmax=vmax
            )
            ax.set_xlabel("X", fontsize=14,fontweight = 'bold')

        # ax.set_aspect('equal')

        # Row labels (first column)
        if j == 0:
            ax.text(-0.45, 0.5, row_labels[i],
                    transform=ax.transAxes,
                    rotation=90, va='center', ha='center',
                    fontsize=14, fontweight='bold')
            ax.set_ylabel("Y", fontsize=14,fontweight = 'bold')
        else:
            ax.tick_params(left=False, labelleft=False)
        # Column titles (top row)
        if i == 0:
            ax.set_title(f"t = {0.01*tsteps[j]}",fontsize = 14)

        row_axes.append(ax)
        ax.tick_params(axis='both', labelsize=14)
        ax.xaxis.set_major_formatter(
            mticker.FormatStrFormatter('%.1f')
        )
        ax.xaxis.set_major_locator(mticker.MultipleLocator(0.5))
        ax.yaxis.set_major_locator(mticker.MultipleLocator(0.5))
        if i == nrow - 1:
            ax.tick_params(labelbottom=True)
            top_row_axes.append(ax)
            # ax.set_xlabel(xlabel, fontsize=12)
                
        else:
            ax.tick_params(bottom=False, labelbottom=False)

        # if i == nrow-1:
        #     top_row_axes.append(ax)
        

    # Colorbar for the row
    cbar = fig.colorbar(im, ax=row_axes, location='right', pad=0.02, fraction=0.03)
    cbar.ax.tick_params(labelsize=14)

# -------------------------
# Add number line below the last row
# -------------------------
ax_left = top_row_axes[0]
ax_right = top_row_axes[-1]

pos_left = ax_left.get_position()
pos_right = ax_right.get_position()
numline_height = 0.025
numline_bottom = pos_left.y0 - 0.09  # slightly below last row

ax_num = fig.add_axes([pos_left.x0, numline_bottom, pos_right.x1 - pos_left.x0, numline_height])


# Number line from 0 to 1
ax_num.set_xlim(0, 2)
ax_num.set_ylim(0, 2)
ax_num.set_yticks([])

sections = [(s, e) for s, e in zip(ts1, ts2)]

for start, end in sections:
    ax_num.axvspan(start, end, color='orange', alpha=0.6)
labels = ['NS']*len(ts1)

for (start, end), label in zip(zip(ts1, ts2), labels):
    x_mid = 0.5 * (start + end)
    ax_num.text(
        x_mid, 0.5,           # centered vertically
        label,
        ha="center",
        va="center",
        fontsize=14,
        color="black",
        fontweight="bold"
    )

grey_intervals = []

# Before first red section
if ts1[0] > 0:
    grey_intervals.append((0, ts1[0]))

# Between red sections
for i in range(len(ts1) - 1):
    if ts2[i] < ts1[i + 1]:
        grey_intervals.append((ts2[i], ts1[i + 1]))

# After last red section
if ts2[-1] < 1:
    grey_intervals.append((ts2[-1], 1))

# Draw grey spans
for start, end in grey_intervals:
    ax_num.axvspan(start, end, color='green', alpha=0.3)
ax_num.set_xlabel("Time", fontsize=14)
ax_num.tick_params(axis='both', labelsize=12)

# plt.tight_layout()
plt.savefig(folder+sample_folder+"/gt_errors.pdf",dpi = 300, bbox_inches='tight')
plt.show()


In [ ]:

# -------------------------------------------------
# GRID + DATA
# -------------------------------------------------
x = np.linspace(0, 1, Nx)
y = np.linspace(0, 1, Ny)
X, Y = np.meshgrid(x, y)

tsteps = [0, 40, 80, 120, 160, 200]

nrow = 4
ncol = len(tsteps)

row_labels = [
    "Ground Truth",
    "AR-FNO",
    "TI-FNO",
    "ANCHOR\n(Ours)",
]

# -------------------------------------------------
# GLOBAL COLOR LIMITS (shared by ALL plots)
# -------------------------------------------------
global_min = min(
    u_test[0][t].min()
    for t in tsteps
)

global_max = max(
    u_test[0][t].max()
    for t in tsteps
)

# -------------------------------------------------
# FIGURE + GRIDSPEC (extra column for colorbar)
# -------------------------------------------------
fig = plt.figure(figsize=(18, 10))

gs = fig.add_gridspec(
    nrow,
    ncol + 1,                       # +1 column for colorbar
    hspace=0.12,
    wspace=0.16,
    width_ratios=[1]*ncol + [0.045]
)

all_axes = []
im_global = None
bottom_row_axes = []

# -------------------------------------------------
# MAIN PANELS
# -------------------------------------------------
for i in range(nrow):
    for j in range(ncol):
        ax = fig.add_subplot(gs[i, j])
        all_axes.append(ax)

        vmin, vmax = global_min, global_max

        if i == 0:
            data = u_test[0][tsteps[j]]
        elif i == 1:
            data = u_pred_auto[0][tsteps[j]]
        elif i == 2:
            data = u_pred_model[0][tsteps[j]]
        else:
            data = u_pred_coup[0][tsteps[j]]

        im = ax.pcolormesh(
            X, Y, data,
            cmap="RdBu",
            shading="gouraud",
            vmin=vmin,
            vmax=vmax
        )

        im_global = im  # store last handle

        # Row labels (first column only)
        if j == 0:
            ax.text(
                -0.45, 0.5,
                row_labels[i],
                transform=ax.transAxes,
                rotation=90,
                va="center",
                ha="center",
                fontsize=14,
                fontweight="bold"
            )
            ax.set_ylabel("Y", fontsize=14, fontweight="bold")
        else:
            ax.tick_params(left=False, labelleft=False)

        # Column titles (top row)
        if i == 0:
            ax.set_title(f"t = {0.01*tsteps[j]:.2f}", fontsize=14)

        # Axis formatting
        ax.tick_params(axis="both", labelsize=14)
        ax.xaxis.set_major_locator(mticker.MultipleLocator(0.5))
        ax.yaxis.set_major_locator(mticker.MultipleLocator(0.5))
        ax.xaxis.set_major_formatter(
            mticker.FormatStrFormatter("%.1f")
        )

        if i == nrow - 1:
            ax.set_xlabel("X", fontsize=14, fontweight="bold")
            bottom_row_axes.append(ax)
        else:
            ax.tick_params(bottom=False, labelbottom=False)

# -------------------------------------------------
# FULL-HEIGHT COLORBAR (spans ALL rows)
# -------------------------------------------------
cax = fig.add_subplot(gs[:, -1])
cbar = fig.colorbar(im_global, cax=cax)
cbar.ax.tick_params(labelsize=14)
# cbar.set_label("Solution value", fontsize=14, fontweight="bold")

# -------------------------------------------------
# NUMBER LINE BELOW LAST ROW
# -------------------------------------------------
pos_left = bottom_row_axes[0].get_position()
pos_right = bottom_row_axes[-1].get_position()

numline_height = 0.025
numline_bottom = pos_left.y0 - 0.085

ax_num = fig.add_axes([
    pos_left.x0,
    numline_bottom,
    pos_right.x1 - pos_left.x0,
    numline_height
])

ax_num.set_xlim(0, 2)
ax_num.set_ylim(0, 2)
ax_num.set_yticks([])

# Red (NS) regions
for s, e in zip(ts1, ts2):
    ax_num.axvspan(s, e, color="orange", alpha=0.6)
    ax_num.text(
        0.5 * (s + e), 0.5,
        "NS",
        ha="center",
        va="center",
        fontsize=14,
        fontweight="bold"
    )

# Blue (non-NS) regions
intervals = []
if ts1[0] > 0:
    intervals.append((0, ts1[0]))
for i in range(len(ts1) - 1):
    if ts2[i] < ts1[i + 1]:
        intervals.append((ts2[i], ts1[i + 1]))
if ts2[-1] < 1:
    intervals.append((ts2[-1], 1))

for s, e in intervals:
    ax_num.axvspan(s, e, color="green", alpha=0.3)

ax_num.set_xlabel("Time", fontsize=14)
ax_num.tick_params(axis="x", labelsize=12)

# -------------------------------------------------
# SAVE
# -------------------------------------------------
plt.savefig(
    folder + sample_folder + "/solutions.pdf",
    dpi=300,
    bbox_inches="tight"
)
plt.show()


In [ ]:
path = '/home/rroy13/jax.venv/DeepONet/'+folder+sample_folder

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import os

# =================================================
# PARAMETERS
# =================================================
# nt = 10
# nx, ny = 50, 50
outdir = path+"/frames"
os.makedirs(outdir, exist_ok=True)
t = np.linspace(0,2,nt)
err1 = np.array(l2_error1)
err2 = np.array(l2_error2)

shaded_spans = []
for arr in mark_coup:
    if len(arr) > 0:
        start = arr[0]
        end = arr[-1]
        shaded_spans.append((start, end))
    
# =================================================
# GRID
# =================================================
x = np.linspace(0, 1, nx)
y = np.linspace(0, 1, ny)
X, Y = np.meshgrid(x, y)

vmin1, vmax1 = u_test[0][30].min(), u_test[0][30].max()   # plots 1–3
vmin2, vmax2 = (np.abs(u_test[0]-u_pred_coup)).min(), (np.abs(u_test[0]-u_pred_model)).max()    # plots 5–6

# =================================================
# TIME LOOP
# =================================================
for ti in range(nt):

    fig = plt.figure(figsize=(14, 10))
    # 5 rows: row0=first plots, row1=colorbars, row2=second plots, row4=timeline
    gs = gridspec.GridSpec(
        nrows=5, ncols=3,
        height_ratios=[1.2, 0.07, 1.2, 0.07, 0.08],
        hspace=0.6, wspace=0.2
    )

    axes = []
    ims = []

    for i in range(6):
        row = (i // 3) * 2  # row 0 for first row, row2 for second
        col = i % 3
        ax = fig.add_subplot(gs[row, col])
        axes.append(ax)

        if i == 0:
            im = ax.pcolormesh(x,y,u_test[0][ti], shading = 'gouraud',
                           cmap="jet", vmin=vmin1, vmax=vmax1)
            ax.set_title("Ground Truth",fontweight = 'bold')
            ax.set_xlabel("X",fontweight = 'bold',fontsize = 12, labelpad=0)
            ax.set_ylabel("Y",fontweight = 'bold',fontsize = 12, labelpad=0)
        elif i == 1:
            im = ax.pcolormesh(x,y,u_pred_model[ti], shading = 'gouraud',
                           cmap="jet", vmin=vmin1, vmax=vmax1)
            ax.set_title("TI-DON",fontweight = 'bold')
            ax.set_xlabel("X",fontweight = 'bold',fontsize = 12, labelpad=0)
            ax.set_ylabel("Y",fontweight = 'bold',fontsize = 12, labelpad=0)
        elif i == 2:
            im = ax.pcolormesh(x,y,u_pred_coup[ti], shading = 'gouraud',
                           cmap="jet", vmin=vmin1, vmax=vmax1)
            ax.set_title("ANCHOR",fontweight = 'bold')
            ax.set_xlabel("X",fontweight = 'bold',fontsize = 12, labelpad=0)
            ax.set_ylabel("Y",fontweight = 'bold',fontsize = 12, labelpad=0)
        elif i == 3:
            # Time-evolving L2 error
            ax.plot(t[:ti+1], err1[:ti+1], color='b', lw=2, label='TI-DON')
            ax.plot(t[:ti+1], err2[:ti+1], color='r', ls='--', lw=2, label='ANCHOR')

            # Shaded spans
            for s, e in zip(ts1, ts2):
                if t[ti] >= s:
                    ax.axvspan(s, min(e, t[ti]), color='g', alpha=0.3)

            ax.set_xlim(t[0], t[-1])
            ax.set_ylim(0, 1.05*max(err1.max(), err2.max()))
            # ax.set_title("L2 Error vs Time")
            ax.set_xlabel("Time",fontweight = 'bold',fontsize = 13)
            ax.set_ylabel(r"Relative $L_2$ Error",fontweight = 'bold',fontsize = 13)
            ax.legend(fontsize=8,loc = 'upper left')
            continue  # skip ims.append

        elif i == 4:
            im = ax.pcolormesh(x,y,np.abs(u_test[0][ti]-u_pred_model[ti]),
                           shading = 'gouraud', cmap="Purples",
                           vmin=vmin2, vmax=vmax2)
            ax.set_title("TI-DON Error",fontweight = 'bold')
            ax.set_xlabel("X",fontweight = 'bold',fontsize = 12, labelpad=0)
            ax.set_ylabel("Y",fontweight = 'bold',fontsize = 12, labelpad=0)
        elif i == 5:
            im = ax.pcolormesh(x,y,np.abs(u_test[0][ti]-u_pred_coup[ti]),
                           shading = 'gouraud', cmap="Purples",
                           vmin=vmin2, vmax=vmax2)
            ax.set_title("ANCHOR Error",fontweight = 'bold')
            ax.set_xlabel("X",fontweight = 'bold',fontsize = 12, labelpad=0)
            ax.set_ylabel("Y",fontweight = 'bold',fontsize = 12, labelpad=0)
            
        if i != 3:
            ims.append(im)

        ax.set_xticks([])
        ax.set_yticks([])
        # ax.set_title(f"Plot {i+1}", fontsize=10)
        ax.tick_params(axis="both", labelsize=12)
        ax.xaxis.set_major_locator(mticker.MultipleLocator(0.5))
        ax.yaxis.set_major_locator(mticker.MultipleLocator(0.5))
        # ax.xaxis.set_major_formatter(
        #     mticker.FormatStrFormatter("%.1f")
        # )

    # -------------------------------------------------
    # COLORBARS (after first row)
    # -------------------------------------------------
    cax1 = fig.add_subplot(gs[1, 0:3])
    cb1 = fig.colorbar(ims[0], cax=cax1, orientation="horizontal")
    # cb1.set_label("Field value")
    cb1.ax.tick_params(labelsize=12)
    cax2 = fig.add_subplot(gs[3, 1:3])
    cb2 = fig.colorbar(ims[4], cax=cax2, orientation="horizontal")
    # cb2.set_label("Absolute error")
    cb2.ax.tick_params(labelsize=12)

    # -------------------------------------------------
    # NUMBER LINE TIMELINE BELOW LAST ROW
    # -------------------------------------------------
    bottom_row_axes = axes[3:6]
    pos_left = bottom_row_axes[0].get_position()
    pos_right = bottom_row_axes[-1].get_position()
    numline_height = 0.022
    numline_bottom = pos_left.y0 - 0.14

    ax_num = fig.add_axes([pos_left.x0, numline_bottom,
                           pos_right.x1 - pos_left.x0, numline_height])
    ax_num.set_xlim(0, 1)
    ax_num.set_ylim(0, 1)
    ax_num.set_yticks([])

    current_t = t[ti]

    # NS regions (magenta)
    for s, e in zip(ts1, ts2):
        if current_t > s:
            ax_num.axvspan(s, min(e, current_t), color="g", alpha=0.3)
            if current_t >= e:
                ax_num.text(0.5*(s+e), 0.5, "NS", ha="center", va="center",
                            fontsize=12, fontweight="bold")

    # non-NS regions (blue)
    intervals = []
    if ts1[0] > 0:
        intervals.append((0, ts1[0]))
    for i_int in range(len(ts1)-1):
        if ts2[i_int] < ts1[i_int+1]:
            intervals.append((ts2[i_int], ts1[i_int+1]))
    if ts2[-1] < 1:
        intervals.append((ts2[-1], 1))
    for s, e in intervals:
        if current_t > s:
            ax_num.axvspan(s, min(e, current_t), color="magenta", alpha=0.5)
            # ax_num.text(
            # 0.5 * (s + e), 0.5,
            # "NS",
            # ha="center",
            # va="center",
            # fontsize=14,
            # fontweight="bold"
            # )

    # moving time marker
    ax_num.plot([current_t, current_t], [0,1], color="k", lw=2)

    ax_num.set_xlabel("Time", fontsize=14)
    ax_num.tick_params(axis="x", labelsize=12)

    # -------------------------------------------------
    # SAVE FRAME
    # -------------------------------------------------
    plt.savefig(f"{outdir}/frame_{ti:03d}.png", dpi=200, bbox_inches="tight")
    if ti == 0 or ti ==0.5:
        plt.show()
    else:
        plt.close(fig)

print("All frames saved.")


In [ ]:
import glob, subprocess
import imageio_ffmpeg as ffmpeg
# print(ffmpeg.get_ffmpeg_exe())  # shows the full path to the bundled ffmpeg binary


# os.chdir('frames/')
# path = os.path
files = sorted(glob.glob(outdir+'/frame_*.png'),
               key=lambda x: int(x.split('me_')[1].split('.png')[0]))
# Create filelist
with open(outdir+'/filelist.txt', 'w') as f:
    for file in files:
        f.write(f"{outdir}/file '{file}'\n")
# cmd = [
#     "ffmpeg", "-y", "-f", "concat", "-safe", "0",
#     "-r", "5", "-i", "filelist.txt",
#     "-vf", "scale=trunc(iw/2)*2:trunc(ih/2)*2",
#     "-c:v", "libx264", "-pix_fmt", "yuv420p", "-crf", "18", "output.mp4"
# ]
# subprocess.run(cmd, check=True)

ffmpeg_path = ffmpeg.get_ffmpeg_exe()

cmd = [
    ffmpeg_path, "-y", "-f", "concat", "-safe", "0",
    "-r", "8", "-i", "filelist.txt",
    "-vf", "scale=trunc(iw/2)*2:trunc(ih/2)*2",
    "-c:v", "libx264", "-pix_fmt", "yuv420p","-preset", "ultrafast", "-crf", "23", outdir+"/output.mp4"
]
subprocess.run(cmd, check=True)
# Run ffmpeg
# !ffmpeg -y -f concat -safe 0 -r 5 -i filelist.txt -vf 'scale=trunc(iw/2)*2:trunc(ih/2)*2' \
#         -c:v libx264 -pix_fmt yuv420p -crf 18 output.mp4

In [ ]:
os.getcwd()